### 3.8.1. Global Optimization

$$
f(\mathbf{x}^\star) \le f(\mathbf{x}) \quad \forall\, \mathbf{x}\in\mathcal{X},
$$
sought for a nonconvex $f$ with several local minimizers.

**Explanation:**

Global optimization seeks the lowest value of a nonconvex objective over the whole feasible set, not merely a point where the gradient vanishes. Because [local methods](../09_Algorithms/11_newtons_method.ipynb) converge to whichever basin they start in, a nonconvex problem may trap them at an inferior local minimizer. Deterministic global methods enumerate and bound regions (as in [branch and bound](./02_branch_and_bound.ipynb)); stochastic methods sample many starting points or maintain a population. For a smooth low-dimensional problem, finding every stationary point and comparing values is exact.

**Intuition:**

The landscape has several valleys; only one is the deepest, and a descent method reports the valley it fell into rather than the deepest one.

![Multimodal landscape with one global minimum](../../Figures/030801_global_optimization_landscape.png)

**Numerical Example:**

$$
f(x) = 3x^4 - 4x^3 - 12x^2.
$$

The stationary points solve $f'(x) = 12x^3 - 12x^2 - 24x = 12x(x-2)(x+1) = 0$, giving $x\in\{-1, 0, 2\}$. Evaluating,
$$
f(-1) = -5\ (\text{local min}),\quad f(0) = 0\ (\text{local max}),\quad f(2) = -32\ (\text{global min}).
$$
A descent method started at $x=-1.5$ would stop at the local minimizer $x=-1$ with value $-5$, missing the global minimizer $x=2$ with value $-32$.

In [1]:
import sympy as sp

x = sp.symbols("x", real=True)
objective = 3*x**4 - 4*x**3 - 12*x**2
first_derivative = sp.diff(objective, x)
second_derivative = sp.diff(objective, x, 2)

stationary_points = sp.solve(first_derivative, x)
classified = [(point,
               "min" if second_derivative.subs(x, point) > 0 else "max",
               objective.subs(x, point))
              for point in stationary_points]
global_minimizer = min((item for item in classified if item[1] == "min"), key=lambda item: item[2])

for location, kind, value in classified:
    print(f"x = {location}  ({kind})  f = {value}")
print("global minimizer:", global_minimizer[0], "value", global_minimizer[2])

x = -1  (min)  f = -5
x = 0  (max)  f = 0
x = 2  (min)  f = -32
global minimizer: 2 value -32


**Equivalent casadi implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x")
objective = 3*decision_variable**4 - 4*decision_variable**3 - 12*decision_variable**2
solver = ca.nlpsol("solver", "ipopt", {"x": decision_variable, "f": objective},
                   {"ipopt.print_level": 0, "print_time": 0, "ipopt.sb": "yes"})

results = {round(float(solver(x0=start)["x"]), 3): round(float(solver(x0=start)["f"]), 3)
           for start in (-1.5, 1.5)}
print("multi-start local minima (minimizer -> value):", results)
print("global minimizer from multi-start:", min(results, key=results.get))

multi-start local minima (minimizer -> value): {-1.0: -5.0, 2.0: -32.0}
global minimizer from multi-start: 2.0


**References:**

[📘 Horst, R., & Tuy, H. (1996). *Global Optimization: Deterministic Approaches* (3rd ed.). Springer.](https://doi.org/10.1007/978-3-662-03199-5)  
[📘 Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.](https://doi.org/10.1007/978-0-387-40065-5)

---

[⬅️ Previous: Distributionally Robust Optimization](../07_Stochastic_and_Robust_Optimization/04_distributionally_robust_optimization.ipynb) | [Next: Branch and Bound ➡️](./02_branch_and_bound.ipynb)